In [82]:
import urllib.error
import urllib.request
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model

import langchain_groq
import os

from dotenv import load_dotenv

load_dotenv("C:\\Users\\socgen\\ML\\agentic_ai_and_ops\\langchain_day5\\.env")

True

In [83]:

model_gr_lamma = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=1000, temperature=0.0)

model_or_paid_gpt40 = init_chat_model("openai/gpt-4o-mini",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_or_free_nvidia = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_or_free = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


In [84]:
booking_requests = [
    "Hi, I'd like 2 tickets for Interstellar at the 7pm show tonight, name is Priya.",
    "can u book me a seat for the 9:30 showing of dune part two? im rohan",
    "URGENT - need to CANCEL my booking for Oppenheimer, confirmation was under Aisha",
]

In [85]:

for msg in booking_requests:
    r = model_gr_lamma.invoke(f"Extract the customer's name, movie, and what they want (book or cancel) from: {msg}")
    print(r.content)
    print("---")

Here are the extracted details:

1. **Customer's name**: Priya
2. **Movie**: Interstellar
3. **What they want**: Book (they want to book 2 tickets)
---
Here are the extracted details:

* Customer's name: Rohan
* Movie: Dune Part Two
* What they want: Book a seat (for the 9:30 showing)
---
Here are the extracted details:

* Customer's name: Aisha
* Movie: Oppenheimer
* What they want: Cancel their booking
---


In [86]:
from pydantic import BaseModel
class MovieShows(BaseModel):
  name : str
  timing:str


In [87]:

response = model_or_free.with_structured_output(MovieShows).invoke("Is Interstellar showing tonight at 7pm at the Downtown cinema ?")

In [88]:
response

MovieShows(name='Interstellar', timing='tonight at 7pm')

In [89]:
from langchain_core.tools import tool

In [90]:
@tool
def check_showtimes(movie_title:str) -> str:
  """Check available showtimes for a movie at the cinema.

  Args:
      movie_title: The exact title of the movie to check
  """
  fake_showtimes = {
      "interstellar": "7:00 PM and 10:15 PM",
      "dune part two": "9:30 PM only",
      "oppenheimer": "Sold out for tonight",
  }
  return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

### @tool decorator will generate the schema of the tools and the tool name and description can be overridden

In [91]:
@tool(
  "book_seats",
  description="Book seats for a movie at the cinema. Provide the movie title, showtime, number of seats, and customer name."
)
def reserve_seats(movie_title:str, showtime:str, num_seats:int, customer_name:str) -> str:
  """Reserve seats for a movie at the cinema.

  Args:
      movie_title: The exact title of the movie to reserve
      showtime: The showtime to reserve seats for
      num_seats: Number of seats to reserve
      customer_name: Name of the customer making the reservation
  """
  return f"Reserved {num_seats} seats for '{movie_title}' at {showtime} under the name {customer_name}."

In [92]:
# Taivily search for showtimes and book seats using the tools

In [93]:
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

In [94]:
import os
from typing import Literal

from tavily import TavilyClient


tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

#args_schema

In [95]:
from pydantic import Field
from typing import Literal

class SeatBookingInput(BaseModel):
    movie_title:str = Field(description='Exact Movie Title')
    seat_count : int = Field(description='Number of seats to book', ge=1, le=10)
    preferred_row : Literal['front', 'middle', 'back'] = Field(default='middle', description='Preferred seat row')

In [96]:
@tool(args_schema=SeatBookingInput)
def book_seats(movie_title:str, seats:int, preferred_row:str)-> str:
  """Book Seats for a Movie"""
  return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

In [97]:
print(book_seats.args)

{'movie_title': {'description': 'Exact Movie Title', 'title': 'Movie Title', 'type': 'string'}, 'seat_count': {'description': 'Number of seats to book', 'maximum': 10, 'minimum': 1, 'title': 'Seat Count', 'type': 'integer'}, 'preferred_row': {'default': 'middle', 'description': 'Preferred seat row', 'enum': ['front', 'middle', 'back'], 'title': 'Preferred Row', 'type': 'string'}}


In [98]:
{'movie_title': {'description': 'Exact Movie Title', 'title': 'Movie Title', 'type': 'string'},
 'seat_count': {'description': 'Number of seats to book', 'maximum': 10, 'minimum': 1,
  'title': 'Seat Count', 'type': 'integer'}, 'preferred_row': {'default': 'middle', 'description': 'Preferred seat row', 'enum': ['front', 'middle', 'back'], 'title': 'Preferred Row', 'type': 'string'}}



{'movie_title': {'description': 'Exact Movie Title',
  'title': 'Movie Title',
  'type': 'string'},
 'seat_count': {'description': 'Number of seats to book',
  'maximum': 10,
  'minimum': 1,
  'title': 'Seat Count',
  'type': 'integer'},
 'preferred_row': {'default': 'middle',
  'description': 'Preferred seat row',
  'enum': ['front', 'middle', 'back'],
  'title': 'Preferred Row',
  'type': 'string'}}

In [99]:
@tool
def book_seats(movie_title:str, seats:int, preferred_row:str,config:str)-> str:
  """Book Seats for a Movie"""
  return f"Booked {seats} seats for {movie_title} in row {preferred_row}"

Never use `config` and `runtime` as args or parameter of Tool.

They are reserved Keywords

In [100]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool
def get_weather(location: str,config:str,units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} {config} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

In [101]:
from langchain.agents import create_agent

agent = create_agent(
    model=model_gr_lamma,
    tools=[get_weather]
    )


In [102]:
# result = agent.invoke(
#         {"messages": [{"role": "user", "content": "What is the weather in Delhi in celsius and tell the forecast?"}]},
# )

In [106]:
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, ToolRuntime
from typing import Any

In [107]:

loyalty_store= InMemoryStore()


@tool
def save_favourite_genres(customer_id:str,genre:str,runtime:ToolRuntime) -> str:
  """Save a customer's facvourite movie genre for future visits"""
  runtime.store.put((customer_id,"preferences"),"favourite_genre",{"value":genre})
  return f"Got it -- I will remmeber you like {genre} movies"

@tool
def recall_favourite_genre(customer_id:str,runtime:ToolRuntime) -> str:
  """ Recall a customer's fav movie genre, if we have saved it before"""
  favourite_genre = runtime.store.get((customer_id,"preferences"),"favourite_genre")
  return favourite_genre.value["value"] if favourite_genre else "We don't have any saved preference for this user"


memory_agent = create_agent(
    model = model_gr_lamma,
    tools=[save_favourite_genres,recall_favourite_genre],
    store=loyalty_store  # Attached to the agent, tools can access it using runtime
)



In [108]:
memory_agent.invoke({"messages": [("user", "Hi, I'm customer priya_01, I love sci-fi movies, please remember that.")]})

{'messages': [HumanMessage(content="Hi, I'm customer priya_01, I love sci-fi movies, please remember that.", additional_kwargs={}, response_metadata={}, id='55cbce10-44ae-4dda-ad84-7328df18e16a'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9ch977npw', 'function': {'arguments': '{"customer_id":"priya_01","genre":"sci-fi"}', 'name': 'save_favourite_genres'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 337, 'total_tokens': 366, 'completion_time': 0.069343796, 'completion_tokens_details': None, 'prompt_time': 0.021896499, 'prompt_tokens_details': None, 'queue_time': 0.05817502, 'total_time': 0.091240295}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9cd1-2e9e-7c12-ae75-6954efce4966-0', tool_calls=[{'name': 'save_favourite_genres', 'args': {'customer_id': 'pri

In [109]:
result = memory_agent.invoke({"messages": [("user", "What genre do I usually like? I'm priya_01.")]})

In [110]:
result

{'messages': [HumanMessage(content="What genre do I usually like? I'm priya_01.", additional_kwargs={}, response_metadata={}, id='36a56961-17e2-4eb5-9f01-6b7cd71f8cd9'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'p4ytr1snr', 'function': {'arguments': '{"customer_id":"priya_01"}', 'name': 'recall_favourite_genre'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 331, 'total_tokens': 352, 'completion_time': 0.043025022, 'completion_tokens_details': None, 'prompt_time': 0.028854121, 'prompt_tokens_details': None, 'queue_time': 0.161789878, 'total_time': 0.071879143}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9cd3-7b55-70a2-9eb5-e63f3daded3b-0', tool_calls=[{'name': 'recall_favourite_genre', 'args': {'customer_id': 'priya_01'}, 'id': 'p4ytr1snr', 'type': 'tool

In [111]:
print(result['messages'][-1].content)

You usually like sci-fi movies.


In [112]:
items = loyalty_store.search(("priya_01", "preferences"))

In [113]:
for item in items:
  print(item)

Item(namespace=['priya_01', 'preferences'], key='favourite_genre', value={'value': 'sci-fi'}, created_at='2026-07-26T05:06:22.772770+00:00', updated_at='2026-07-26T05:06:22.772772+00:00', score=None)


In [114]:
@tool(return_direct=True)
def get_exact_refund_policy() -> str:
    """Tell the refund policy."""
    return "Tickets are refundable up to 2 hours before showtime. No refunds after that."

direct_agent = create_agent(model=model_gr_lamma, tools=[get_exact_refund_policy])
result = direct_agent.invoke({"messages": [("user", "What's your refund policy? Please explain in points")]})
print(result["messages"][-1].content)

Tickets are refundable up to 2 hours before showtime. No refunds after that.


# Dynamic Tool Loading & Calling

In [115]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@tool
def standard_booking(movie_title: str) -> str:
    """Book a standard seat."""
    return f"Standard seat booked for {movie_title}."

@tool
def vip_lounge_booking(movie_title: str) -> str:
    """Book a VIP lounge seat with premium service. VIP members only."""
    return f"VIP lounge seat booked for {movie_title}."


gated_agent = create_agent(
    model=model_gr_lamma,
    tools=[standard_booking]
)

In [116]:
result_regular = gated_agent.invoke({"messages": [("user", "Book me a VIP lounge seat for Dune")]})


In [126]:
result_regular


{'messages': [HumanMessage(content='Book me a VIP lounge seat for Dune', additional_kwargs={}, response_metadata={}, id='fb44b9da-53cc-4042-b7fc-51a8babedd54'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '06ca22zys', 'function': {'arguments': '{"movie_title":"Dune"}', 'name': 'standard_booking'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 219, 'total_tokens': 236, 'completion_time': 0.048735691, 'completion_tokens_details': None, 'prompt_time': 0.013230781, 'prompt_tokens_details': None, 'queue_time': 0.051940143, 'total_time': 0.061966472}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9cf1-f7a5-7ff2-a9d5-fb66964fd11c-0', tool_calls=[{'name': 'standard_booking', 'args': {'movie_title': 'Dune'}, 'id': '06ca22zys', 'type': 'tool_call'}], invalid_tool_calls=

In [135]:
result_regular.get("messages")[0].content

'Book me a VIP lounge seat for Dune'